In [3]:
import requests
from bs4 import BeautifulSoup
import random
import time
import re
import pandas as pd
from tqdm import tqdm
import os

os.makedirs('data', exist_ok=True)

# ===================== 配置 =====================
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
]

BASE_URL = "https://movie.douban.com/top250"
MAX_RETRY = 2
DELAY_MIN = 1
DELAY_MAX = 3

# ===================== 工具函数 =====================
def get_header():
    return {"User-Agent": random.choice(USER_AGENTS)}

def random_delay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def fetch_html(url):
    for i in range(MAX_RETRY):
        try:
            resp = requests.get(url, headers=get_header(), timeout=15)
            resp.raise_for_status()
            random_delay()
            return resp.text
        except Exception as e:
            time.sleep(2)
    return None

# ===================== 【完美匹配你截图】解析函数 =====================
def parse_movie_item(item):
    try:
        # 1 排名
        rank = item.find("em").text.strip()

        # 2 标题
        titles = item.find_all("span", class_="title")
        title_cn = titles[0].text.strip()
        title_en = titles[1].text.strip().replace("/ ", "") if len(titles) > 1 else "无"

        # 3 评分 & 评价人数
        rating = item.find("span", class_="rating_num").text.strip()
        vote_str = item.find(string=re.compile(r"\d+人评价")).strip()
        vote_num = re.findall(r"\d+", vote_str)[0]

        # 4 导演 & 主演（干净版）
        info_p = item.find("div", class_="bd").p
        info_text = info_p.text.strip().replace("\xa0", " ") if info_p else "无信息"
        director_actor = re.split(r'\d{4}', info_text)[0].strip()

        # =====================
        # 【重点】按你截图精准提取简介
        # =====================
        quote = "无简介"
        bd_div = item.find("div", class_="bd")
        
        # 完全按照你截图的结构：<p class="quote"> <span> 简介 </span> </p>
        quote_tag = bd_div.find("p", class_="quote")
        if quote_tag:
            span_tag = quote_tag.find("span")
            if span_tag:
                quote = span_tag.text.strip()

        # 6 链接
        link = item.find("a")["href"]

        return {
            "排名": rank,
            "中文标题": title_cn,
            "英文标题": title_en,
            "评分": rating,
            "评价人数": vote_num,
            "导演&主演": director_actor,
            "简介": quote,
            "详情链接": link
        }
    except:
        return None

# ===================== 爬取主逻辑 =====================
def crawl_douban_top250():
    all_movies = []
    print("开始爬取 豆瓣电影Top250...")

    for start in tqdm(range(0, 250, 25), desc="爬取进度"):
        url = f"{BASE_URL}?start={start}"
        html = fetch_html(url)
        if not html:
            continue
        
        soup = BeautifulSoup(html, "html.parser")
        items = soup.find_all("div", class_="item")

        for item in items:
            data = parse_movie_item(item)
            if data:
                all_movies.append(data)

    return all_movies

# ===================== 执行 =====================
if __name__ == "__main__":
    movie_data = crawl_douban_top250()
    df = pd.DataFrame(movie_data)

    # 关闭文件再运行！
    df.to_csv("data/douban_top250_final333.csv", index=False, encoding="utf-8-sig")

    print(f"\n✅ 爬取完成！共 {len(df)} 部电影")
    print("✅ 简介已按列表页正常提取！")

开始爬取 豆瓣电影Top250...


爬取进度: 100%|████████████████████████████████████████████████████████████████████████| 10/10 [00:34<00:00,  3.41s/it]


✅ 爬取完成！共 250 部电影
✅ 简介已按列表页正常提取！
